In [1]:
import cv2
import mediapipe as mp
import time
import numpy as np

In [2]:
import PythonToFLStudioBridge as PTSB


In [17]:

class handDetector():
    def __init__(self, mode=False, maxHands=2, detectionCon=0.5, trackCon=0.5):
        self.mode = mode
        self.maxHands = maxHands
        self.detectionCon = detectionCon
        self.trackCon = trackCon
        
        self.mpHands = mp.solutions.hands
        self.hands = self.mpHands.Hands(static_image_mode = self.mode, 
                                        max_num_hands = self.maxHands,
                                        min_detection_confidence = self.detectionCon, 
                                        min_tracking_confidence = self.trackCon)
        self.mpDraw = mp.solutions.drawing_utils

    def findHands(self, img, draw=True):
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.results = self.hands.process(imgRGB)
        #print(results.multi_hand_landmarks)
        
        if self.results.multi_hand_landmarks:
            for handLms in self.results.multi_hand_landmarks:
                if draw: self.mpDraw.draw_landmarks(img, handLms, 
                                                    self.mpHands.HAND_CONNECTIONS)
        return img, self.results.multi_hand_landmarks

    def findPosition(self, img, handNo=0, draw=True, axis=False):
        
        if axis:
            height, width, _ = img.shape
            center_x, center_y = width // 2, height // 2
            cv2.line(img, (center_x, 0), (center_x, height), (0, 255, 0), 2)  # Y-axis
            cv2.line(img, (0, center_y), (width, center_y), (255, 0, 0), 2)  # X-axis

            # Display the min and max coordinates for the axes
            cv2.putText(img, f"(0, {center_y})", (10, center_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)  # Min X
            cv2.putText(img, f"({width}, {center_y})", (width - 150, center_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)  # Max X
            cv2.putText(img, f"({center_x}, 0)", (center_x + 10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)  # Min Y
            cv2.putText(img, f"({center_x}, {height})", (center_x + 10, height - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1, cv2.LINE_AA)  # Max Y

        
        
        
        lmList = []
        if self.results.multi_hand_landmarks:
            myHand = self.results.multi_hand_landmarks[handNo]
            
            for id, lm in enumerate(myHand.landmark):
                # print(id, lm)
                h,w,c = img.shape
                
                cx, cy = int(lm.x*w), int(lm.y*h)
                # cx0, cy0 = int(myHand.landmark[8].x*w), int(myHand.landmark[8].y*h)
                lmList.append([id, cx, cy])
                if draw:
                    cv2.circle(img, (cx, cy), 25, (255,255,255), cv2.FILLED)
                # cv2.circle(img, (cx0, cy0), 15, (255,255,0), cv2.FILLED)
        return lmList
    
    def Length(self, points):
        print(points)
        point = np.subtract(points[0], points[1])**2
        length = np.sqrt(point[0]+point[1])
        print(f"length {length}")
        return length
    
    def PointConnect(self, img, pointA, pointB, middle):
        cv2.circle(img, pointA, 10, (0, 255, 0), -1)  # Green circle for thumb tip
        cv2.circle(img, pointB, 10, (0, 0, 255), -1)  # Red circle for index tip
        
        cv2.line(img, pointA, middle, (255, 255, 255), 2)  # Line from thumb to midpoint
        cv2.line(img, middle, pointB, (255, 255, 255), 2)  # Line from midpoint to index tip
        
        cv2.circle(img, middle, 10, (255, 0, 0), -1)  # Blue circle for midpoint
        
    def Index_Thumb(self, img, draw=False, draw_interval=False):
        mid = []
        if self.results and self.results.multi_hand_landmarks:
            for hand in self.results.multi_hand_landmarks:
                if len(hand.landmark) > 7:
                    index = hand.landmark[8]
                    thumb = hand.landmark[4]
                    
                    h, w, c = img.shape
                    index_x, index_y = int(index.x * w), int(index.y * h)
                    thumb_x, thumb_y = int(thumb.x * w), int(thumb.y * h)
                    
                    middle_x = (index_x + thumb_x) // 2
                    middle_y = (index_y + thumb_y) // 2
                    
                    if draw: self.PointConnect(img, (thumb_x, thumb_y), (index_x, index_y), (middle_x, middle_y))
                    
                    mid.append((middle_x, middle_y))
            if len(mid)==2 and draw_interval: 
                cv2.line(img, mid[0], mid[1], (255,255,255), 2)
                length = self.Length(mid)
                PTSB.SendCC(percent=length/356, ctrl=2)
        return mid
    
    def hand_direction_bool(self, img, draw=False):
        if self.results and self.results.multi_hand_landmarks:
            for hand in self.results.multi_hand_landmarks:
                if len(hand.landmark) > 7:
                    index = hand.landmark[8]
                    thumb = hand.landmark[4]
                    
                    h, w, c = img.shape
                    index_x, index_y = int(index.x * w), int(index.y * h)
                    thumb_x, thumb_y = int(thumb.x * w), int(thumb.y * h)
                    
                    if thumb_x < index_x:
                        return True
                    else:
                        return False
        return None
    
    

In [24]:
pTime = 0
cTime = 0
cap = cv2.VideoCapture(0)
detector = handDetector()
draw=False
draw_interval=False
axis=False
while True:
    #time.sleep(1)
    counter  = 0
    success, img = cap.read()
    img, shit = detector.findHands(img)
    lmList = detector.findPosition(img, draw=False, axis=axis)
    # print(detector.hand_direction_bool(img, draw=draw)) #we should use this later for now just TODO
    print([0 if shit is None else len(shit)])
    middle_points = detector.Index_Thumb(img, draw=draw, draw_interval=draw_interval)
    # if len(lmList)!=0: print(lmList[0])
        
    # if len(lmList)>7: print(lmList[8])
    if (len(lmList)>2) and counter%5==0: 
        percents = PTSB.data_writer(data=lmList[8][1:])
        # print(percents[1]) #y coord
        PTSB.SendCC(percent=percents[1], ctrl=1)
        counter+=1
        
    cTime = time.time()
    fps = 1/(cTime-pTime)
    pTime = cTime
        
    cv2.putText(img, str(int(fps)), (10,10), cv2.FONT_HERSHEY_SCRIPT_SIMPLEX,
                1, (255,255,255), 3)
    
    cv2.putText(img, str(middle_points), (10,30), cv2.FONT_HERSHEY_SCRIPT_SIMPLEX, #len(lmList)
                0.5, (255,255,255), 1)
        
    cv2.imshow("Image", img)
    #cv2.waitKey(1)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    if cv2.waitKey(1) & 0xFF == ord('d'): draw=~draw
    if cv2.waitKey(1) & 0xFF == ord('i'): draw_interval=~draw_interval
    if cv2.waitKey(1) & 0xFF == ord('a'): axis=~axis
cap.release()
cv2.destroyAllWindows()

[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[0]
[1]
Sent CC message: control_change channel=0 control=1 value=53 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=64 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=65 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=66 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=68 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=68 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=68 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=69 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=69 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=69 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=69 time=0
[2]
Sent CC message: control_change channel=0 control=1 value=70 time=0
[2]
Sent CC message: control_cha

In [1]:
# import mido
# from mido import Message

In [28]:
# import time

In [9]:
# virtual_midi_port_name = "LoopMIDI Port 1"

In [10]:
# print("Available Output Ports:", mido.get_output_names())

Available Output Ports: ['Microsoft GS Wavetable Synth 0', 'loopMIDI Port 1', 'loopMIDI Port 1 2']


In [32]:
# with mido.open_output(mido.get_output_names()[1]) as outport:
#     print(f"Sending signals to: {virtual_midi_port_name}")
    
#     while True:
#         try:
#             percent = float(input("Enter a percentage value (0-100): "))
#             if percent < 0 or percent > 100:
#                 print("Please enter a value between 0 and 100.")
#                 continue
            
#             # Convert percentage (0-100) to MIDI value (0-127)
#             midi_value = int((percent / 100) * 127)
            
#             # Create and send a Control Change (CC) message
#             cc_message = Message('control_change', control=1, value=midi_value)  # CC #1 is commonly used
#             outport.send(cc_message)
#             print(f"Sent CC message: {cc_message}")
#             time.sleep(1)

#         except KeyboardInterrupt:
#             print("\nExiting...")
#             break

Sending signals to: LoopMIDI Port
Sent CC message: control_change channel=0 control=1 value=63 time=0
Sent CC message: control_change channel=0 control=1 value=127 time=0
Sent CC message: control_change channel=0 control=1 value=101 time=0
Sent CC message: control_change channel=0 control=1 value=41 time=0


ValueError: could not convert string to float: ''

In [19]:
# with mido.open_output(mido.get_output_names()[1]) as outport:
#     print(f"Successfully opened port: {outport}")

Successfully opened port: <open output 'loopMIDI Port 1' (RtMidi/WINDOWS_MM)>
